# Topic Classification: movie / restaurant / book

Zero-shot topic classification using a pre-trained
facebook/bart-large-mnli model.
No training data required.

Run the notebook from top to bottom. N.B. The first zero-shot run downloads BART (approx. 1.6 GB).

## 1. Setup

In [15]:
import pandas as pd
import sklearn
from sklearn.metrics import classification_report

LABELS = ["movie", "restaurant", "book"]
print(f"pandas {pd.__version__} sklearn {sklearn.__version__}")

pandas 2.2.2 sklearn 1.6.1


## 2. Load the test set

The test set file is tab-separated (TSV) (Columns: sentence id, text, sentiment, topic). topic will be used as ground truth and sentiment will be ignored.

In [5]:
df = pd.read_csv("../Sentiment-topic-test.tsv", sep="\t")
df.columns = [c.strip() for c in df.columns]

print("shape:", df.shape)
print(df["topic"].value_counts().to_string())

shape: (10, 4)
topic
movie         5
restaurant    3
book          2


## 3. Zero-shot classification (facebook/bart-large-mnli)

Each sentence is scored against the three topics posed as hypotheses. The top-scoring
topic is the prediction. device=0 uses the GPU (use -1 to change to CPU).

In [7]:
import torch
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0) #GPU

template = "The topic of this text is a {}."
df["pred_zeroshot"] = [
    classifier(t, LABELS, hypothesis_template=template)["labels"][0]
    for t in df["text"]
]

correct = (df["pred_zeroshot"] == df["topic"]).sum()
total = len(df)
accuracy = correct / total
print("Zero-shot accuracy:", round(accuracy, 3), f"({correct}/{total})")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Zero-shot accuracy: 1.0 (10/10)


## 4. Evaluation

In [9]:
print(classification_report(df["topic"], df["pred_zeroshot"], labels=LABELS, zero_division=0))

print("Predictions vs gold:")
correct = 0
for i in range(len(df)):
    gold = df["topic"][i]
    pred = df["pred_zeroshot"][i]
    if pred == gold:
        correct += 1
        result = "correct"
    else:
        result = "wrong"
    print(i, "-", result, "| gold:", gold, "| pred:", pred, "|", df["text"][i][:50])

print(f"{correct} out of {len(df)} are correct")

              precision    recall  f1-score   support

       movie       1.00      1.00      1.00         5
  restaurant       1.00      1.00      1.00         3
        book       1.00      1.00      1.00         2

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

Predictions vs gold:
0 - correct | gold: movie | pred: movie | It took eight years for Warner Brothers to recover
1 - correct | gold: restaurant | pred: restaurant | All the New York University students love this din
2 - correct | gold: restaurant | pred: restaurant | This Italian place is really trendy but they have 
3 - correct | gold: book | pred: book | In conclusion, my review of this book would be: I 
4 - correct | gold: movie | pred: movie | The story of this movie is focused on Carl Brashea
5 - correct | gold: movie | pred: movie | Chris O'Donnell stated that while filming for this
6 - correct | gold: re

## Save results

In [11]:
results = df[["sentence id", "text", "sentiment", "topic", "pred_zeroshot"]].rename(
    columns={"topic": "gold_topic"})
results.to_csv("../content/topic_analysis_results.csv", index=False)